# La API en un entorno empresarial: un nivel por decisión de negocio

`api/main.py` sirve un modelo LightGBM independiente por nivel de la jerarquía M5
(ver `config/levels.py`). Este notebook se conecta a la API **ya levantada con
Docker** (`make docker-api`, puerto 8000) y simula, para tres niveles, cómo un
sistema externo pediría una predicción y tomaría una decisión simple con ella.

A diferencia de una versión anterior de este notebook, acá no se toca ningún
artifact de entrenamiento ni se manda un vector de features ya calculado: cada
escenario manda `series_id` + fecha (más lo que ya se sabe de antemano de ese
día -- SNAP, precio, calendario de eventos) al endpoint `/predict/{level_id}/forecast`,
y es la API la que reconstruye lags/rolling/target-encoding sobre la historia
real de esa serie (`api/feature_builder.py`). El valor real observado para
comparar se lee directo de `data/processed/`, no de un split de test.

In [4]:
import pandas as pd
import requests
from loguru import logger

import config

pd.set_option("display.max_columns", None)

BASE_URL = "http://localhost:8000"

try:
    requests.get(f"{BASE_URL}/health", timeout=2).raise_for_status()
except requests.exceptions.ConnectionError:
    raise RuntimeError("La API no responde en :8000 -- levantala con `make docker-api`")

logger.success("API OK en {}", BASE_URL)

2026-09-17 01:01:28.267 | SUCCESS  | __main__:<module>:16 - API OK en http://localhost:8000


## Helper: predecir una serie contra la API (con datos reales, no un vector precalculado)

Lee de `data/processed/` (no de un artifact de entrenamiento) el último día con
dato real de esa serie: su venta real -- para comparar al final -- y lo que ya
se sabía de antemano para ese día (SNAP, precio, evento), que se manda como
`overrides`. Todo lo demás (~190 columnas de lags/rolling/encoding) lo
reconstruye la API sola a partir de la historia real de la serie.

In [5]:
def predict(level_id: int, series_id: str, grain: str = "daily") -> dict:
    level = config.LEVELS_BY_ID[level_id]
    cols = ["series_id", "date", "sales", "avg_sell_price", "snap",
            "event_name_1", "event_type_1", "event_name_2", "event_type_2"]
    df = pd.read_parquet(config.dataset_level_path(level, grain), columns=cols)
    last = df[df["series_id"] == series_id].sort_values("date").iloc[-1]
    date = pd.Timestamp(last["date"]).date()

    # Lo único que la API no puede derivar de la historia de la serie: lo que
    # ya se sabía de antemano para el día pedido (SNAP, precio, calendario de
    # eventos). Sin overrides, /forecast asume "sin evento" y arrastra el
    # último precio conocido -- acá sí conocemos el real, así que lo mandamos.
    overrides = {
        "avg_sell_price": float(last["avg_sell_price"]) if pd.notna(last["avg_sell_price"]) else None,
        "snap": int(last["snap"]),
        "event_name_1": last["event_name_1"], "event_type_1": last["event_type_1"],
        "event_name_2": last["event_name_2"], "event_type_2": last["event_type_2"],
    }
    overrides = {k: v for k, v in overrides.items() if pd.notna(v)}

    resp = requests.post(
        f"{BASE_URL}/predict/{level_id}/forecast?grain={grain}",
        json={"series_id": series_id, "date": str(date), "overrides": overrides},
    )
    resp.raise_for_status()

    pred = resp.json()["prediction"]
    real = float(last["sales"])
    return {"series_id": series_id, "date": date, "prediccion": round(pred, 1), "real": real,
            "error_pct": round(100 * (pred - real) / real, 1)}

## Escenario 1 -- Nivel 1 (total): guidance financiero

Finanzas quiere una proyección de ventas de la cadena completa antes del cierre.

In [6]:
predict(1, "TOTAL")

{'series_id': 'TOTAL',
 'date': datetime.date(2016, 5, 22),
 'prediccion': 48259.4,
 'real': 54338.0,
 'error_pct': -11.2}

## Escenario 2 -- Nivel 3 (categoría): plan de compras con colchón de seguridad

Category management pide la orden de compra: pronóstico + `SAFETY_STOCK` para no
quedarse corto ante un pico no capturado por el modelo.

In [7]:
SAFETY_STOCK = 1.15

df_categoria = pd.DataFrame([predict(3, cat) for cat in ["FOODS", "HOBBIES", "HOUSEHOLD"]])
df_categoria["orden_sugerida"] = (df_categoria["prediccion"] * SAFETY_STOCK).round(0)
df_categoria

,series_id,date,prediccion,real,error_pct,orden_sugerida
0,FOODS,2016-05-22,32358.2,35967.0,-10.0,37212.0
1,HOBBIES,2016-05-22,5157.3,5280.0,-2.3,5931.0
2,HOUSEHOLD,2016-05-22,12832.9,13091.0,-2.0,14758.0


## Escenario 3 -- Nivel 6 (tienda): ¿turno extra de caja?

Regla simple: si la predicción supera `UMBRAL_TURNO_EXTRA` unidades/día, se
refuerza personal.

In [ ]:
UMBRAL_TURNO_EXTRA = 6_000

df_tienda = pd.DataFrame([predict(6, tienda) for tienda in ["CA_1_CA", "CA_2_CA", "CA_3_CA", "CA_4_CA"]])
df_tienda["turno_extra"] = df_tienda["prediccion"] > UMBRAL_TURNO_EXTRA
df_tienda